In [1]:
# CREDIT CARD CUSTOMER AND TRANSACTION ANALYTICS
#
# Business problem:
# The bank wants to understand how customers use their credit cards, how they pay,
# how much of their credit limit they use and which customers may be risky.
#
# Business objectives:
# 1. Measure transaction volume and transaction value
# 2. Find the customer segments and merchant categories that spend the most
# 3. Study monthly trends, credit limits, utilization, payments and balances
# 4. Find high value customers, high risk customers and customers to target
#
# Data: Kaggle "computingvictor/transactions-fraud-datasets" (2010 to 2019)
# We only study CREDIT cards in this project.

import pandas as pd
import json

In [2]:
# STEP 1: LOOK AT THE DATA
# Load the customers file
customers_data = pd.read_csv("Raw_Data/users_data.csv")

print("Customers:", customers_data.shape)
customers_data.head()

Customers: (2000, 14)


,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [3]:
# Check for missing values and duplicates in customers
print("Missing values:", customers_data.isnull().sum().sum())
print("Duplicate rows:", customers_data.duplicated().sum())

Missing values: 0
Duplicate rows: 0


In [4]:
# Load the cards file
cards_data = pd.read_csv("Raw_Data/cards_data.csv")

print("Cards:", cards_data.shape)
cards_data.head()

Cards: (6146, 13)


,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [5]:
# Check the cards file
print("Missing values:", cards_data.isnull().sum().sum())
print("Duplicate rows:", cards_data.duplicated().sum())
print()
print(cards_data["card_type"].value_counts())

Missing values: 0
Duplicate rows: 0

card_type
Debit              3511
Credit             2057
Debit (Prepaid)     578
Name: count, dtype: int64


In [6]:
# Load the merchant category names
mcc_file = open("Raw_Data/mcc_codes.json")
mcc_codes = json.load(mcc_file)
mcc_file.close()

print("Merchant categories:", len(mcc_codes))

Merchant categories: 109


In [7]:
# The transactions file is very big (1.2 GB), so we only look at the first 5 rows here
transactions_preview = pd.read_csv("Raw_Data/transactions_data.csv", nrows=5)
transactions_preview

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [8]:
# STEP 2: CLEAN THE DATA
# CUSTOMERS
customers_clean = customers_data.copy()
customers_clean = customers_clean.rename(columns={"id": "customer_id", "current_age": "age"})

# Remove the $ sign from money columns and change them to numbers
customers_clean["yearly_income"] = customers_clean["yearly_income"].str.replace("$", "").astype(float)
customers_clean["per_capita_income"] = customers_clean["per_capita_income"].str.replace("$", "").astype(float)
customers_clean["total_debt"] = customers_clean["total_debt"].str.replace("$", "").astype(float)

# Some incomes are below $1,000, which is not realistic. We use the median income for them
median_income = customers_clean["yearly_income"].median()
customers_clean.loc[customers_clean["yearly_income"] < 1000, "yearly_income"] = median_income

# Remove columns we do not need
customers_clean = customers_clean.drop(columns=["address", "birth_year", "birth_month", "retirement_age", "latitude", "longitude"])

customers_clean.head()

,customer_id,age,gender,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,Female,29278.0,59696.0,127613.0,787,5
1,1746,53,Female,37891.0,77254.0,191349.0,701,5
2,1718,81,Female,22681.0,33483.0,196.0,698,5
3,708,63,Female,163145.0,249925.0,202328.0,722,4
4,1164,43,Male,53797.0,109687.0,183855.0,675,1


In [9]:
# Make customer groups
customers_clean["age_group"] = pd.cut(customers_clean["age"], bins=[0, 29, 44, 59, 200],
                                      labels=["18 to 29", "30 to 44", "45 to 59", "60 plus"])

customers_clean["income_segment"] = pd.cut(customers_clean["yearly_income"], bins=[0, 30000, 60000, 100000, 10000000],
                                           labels=["Low Income", "Middle Income", "Upper Middle Income", "High Income"])

customers_clean["credit_score_band"] = pd.cut(customers_clean["credit_score"], bins=[0, 579, 669, 739, 799, 850],
                                              labels=["Poor", "Fair", "Good", "Very Good", "Excellent"])

# Debt to income = total debt divided by yearly income
customers_clean["debt_to_income"] = (customers_clean["total_debt"] / customers_clean["yearly_income"]).round(2)
customers_clean["debt_band"] = pd.cut(customers_clean["debt_to_income"], bins=[-1, 0.5, 1.5, 2.5, 1000],
                                      labels=["Low Debt", "Moderate Debt", "High Debt", "Very High Debt"])

customers_clean["age_group"].value_counts()

age_group
45 to 59    543
30 to 44    531
18 to 29    480
60 plus     446
Name: count, dtype: int64

In [10]:
# CARDS
cards_clean = cards_data.copy()
cards_clean = cards_clean.rename(columns={"id": "card_id", "client_id": "customer_id"})

# Keep only credit cards
cards_clean = cards_clean[cards_clean["card_type"] == "Credit"]

# Remove the $ sign from credit limit
cards_clean["credit_limit"] = cards_clean["credit_limit"].str.replace("$", "").astype(float)

# A credit card cannot have a $0 limit, so we use the median limit for those cards
median_limit = cards_clean[cards_clean["credit_limit"] > 0]["credit_limit"].median()
cards_clean.loc[cards_clean["credit_limit"] == 0, "credit_limit"] = median_limit

# Year the account was opened
cards_clean["open_year"] = pd.to_datetime(cards_clean["acct_open_date"], format="%m/%Y").dt.year

# Credit limit groups
cards_clean["limit_band"] = pd.cut(cards_clean["credit_limit"], bins=[0, 5000, 10000, 15000, 20000, 1000000],
                                   labels=["Under 5K", "5K to 10K", "10K to 15K", "15K to 20K", "Above 20K"])

# Remove card number, CVV and other columns we do not need
cards_clean = cards_clean.drop(columns=["card_number", "cvv", "expires", "card_on_dark_web", "card_type",
                                        "year_pin_last_changed", "acct_open_date"])

print("Credit cards:", len(cards_clean))
cards_clean.head()

Credit cards: 2057


,card_id,customer_id,card_brand,has_chip,num_cards_issued,credit_limit,open_year,limit_band
3,42,825,Visa,NO,1,12400.0,2003,10K to 15K
5,4537,1746,Visa,YES,1,27500.0,2003,Above 20K
15,281,708,Visa,YES,2,98100.0,2011,Above 20K
20,27,68,Visa,YES,2,9900.0,2002,5K to 10K
22,5435,68,Visa,YES,2,15800.0,2005,15K to 20K


In [11]:
# TRANSACTIONS
# The file has 13 million rows. It is too big for this laptop, so we read 1 million rows at a time
# and keep only the credit card transactions.
credit_card_ids = cards_clean["card_id"]
columns_to_read = ["id", "date", "client_id", "card_id", "amount", "use_chip", "merchant_state", "mcc", "errors"]
pieces = []

for chunk in pd.read_csv("Raw_Data/transactions_data.csv", usecols=columns_to_read, chunksize=1000000):
    chunk = chunk[chunk["card_id"].isin(credit_card_ids)]
    pieces.append(chunk)

transactions_clean = pd.concat(pieces)
print("Credit card transactions:", len(transactions_clean))

Credit card transactions: 4109189


In [12]:
# Clean the transactions
transactions_clean = transactions_clean.rename(columns={"id": "transaction_id", "client_id": "customer_id"})

# Remove the $ sign from amount
transactions_clean["amount"] = transactions_clean["amount"].str.replace("$", "").astype(float)

# Change date text to a real date and keep the month and year
transactions_clean["date"] = pd.to_datetime(transactions_clean["date"])
transactions_clean["month_start"] = transactions_clean["date"].dt.to_period("M").dt.to_timestamp()
transactions_clean["year"] = transactions_clean["date"].dt.year

# Channel: Swipe, Chip or Online
transactions_clean["channel"] = transactions_clean["use_chip"].str.replace(" Transaction", "")

transactions_clean.head()

,transaction_id,date,customer_id,card_id,amount,use_chip,merchant_state,mcc,errors,month_start,year,channel
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,IA,5311,NaN,2010-01-01,2010,Swipe
16,7475344,2010-01-01 00:32:00,646,2093,73.79,Swipe Transaction,PA,7538,NaN,2010-01-01,2010,Swipe
20,7475348,2010-01-01 00:36:00,1376,2182,54.27,Swipe Transaction,TX,5813,NaN,2010-01-01,2010,Swipe
21,7475349,2010-01-01 00:37:00,1682,238,20.74,Swipe Transaction,ND,5411,NaN,2010-01-01,2010,Swipe
24,7475353,2010-01-01 00:43:00,301,3742,10.17,Online Transaction,NaN,4784,NaN,2010-01-01,2010,Online


In [13]:
# Location: online purchases have no state, foreign shops have a country name instead of a state code
transactions_clean["merchant_state"] = transactions_clean["merchant_state"].fillna("Online")

transactions_clean["location_type"] = "USA"
transactions_clean.loc[transactions_clean["merchant_state"].str.len() > 2, "location_type"] = "International"
transactions_clean.loc[transactions_clean["merchant_state"] == "Online", "location_type"] = "Online"

transactions_clean["location_type"].value_counts()

location_type
USA              3610209
Online            470847
International      28133
Name: count, dtype: int64

In [14]:
# Errors: a transaction with an error was declined
transactions_clean["is_failed"] = transactions_clean["errors"].notnull()
transactions_clean["is_insufficient_balance"] = transactions_clean["errors"].fillna("").str.contains("Insufficient Balance")

print("Failed transactions:", transactions_clean["is_failed"].sum())
print("Refunds (negative amounts):", (transactions_clean["amount"] < 0).sum())
print("Duplicate transaction ids:", transactions_clean["transaction_id"].duplicated().sum())

Failed transactions: 61828
Refunds (negative amounts): 216832


Duplicate transaction ids: 0


In [15]:
# Keep only customers and cards that have transactions
customers_clean = customers_clean[customers_clean["customer_id"].isin(transactions_clean["customer_id"])]
cards_clean = cards_clean[cards_clean["card_id"].isin(transactions_clean["card_id"])]

print("Customers:", len(customers_clean))
print("Credit cards:", len(cards_clean))

Customers: 855
Credit cards: 1351


In [16]:
# CLEANING SUMMARY
print("1. Customers: removed $ signs, fixed 8 unrealistic incomes, removed address and birth columns")
print("2. Customers: added age group, income segment, credit score band and debt band")
print("3. Cards: kept only credit cards, fixed 26 cards with a $0 limit, removed card number and CVV")
print("4. Transactions: kept only credit card transactions, removed $ signs, changed dates")
print("5. Transactions: filled empty states with Online and added location type")
print("6. Transactions: marked failed transactions; failed ones and refunds do not count as spending")
print("7. Kept", len(customers_clean), "customers and", len(cards_clean), "credit cards that have transactions")

1. Customers: removed $ signs, fixed 8 unrealistic incomes, removed address and birth columns
2. Customers: added age group, income segment, credit score band and debt band
3. Cards: kept only credit cards, fixed 26 cards with a $0 limit, removed card number and CVV
4. Transactions: kept only credit card transactions, removed $ signs, changed dates
5. Transactions: filled empty states with Online and added location type
6. Transactions: marked failed transactions; failed ones and refunds do not count as spending
7. Kept 855 customers and 1351 credit cards that have transactions


In [17]:
# STEP 3: NEW COLUMNS
# Only successful purchases count as spending
transactions_clean["purchase_count"] = 0
transactions_clean["purchase_value"] = 0.0
transactions_clean["refund_value"] = 0.0

good_purchase = (transactions_clean["amount"] > 0) & (transactions_clean["is_failed"] == False)
good_refund = (transactions_clean["amount"] < 0) & (transactions_clean["is_failed"] == False)

transactions_clean.loc[good_purchase, "purchase_count"] = 1
transactions_clean.loc[good_purchase, "purchase_value"] = transactions_clean["amount"]
transactions_clean.loc[good_refund, "refund_value"] = transactions_clean["amount"] * -1

# Online purchases, failed transactions and insufficient balance errors (as 1 or 0)
transactions_clean["online_count"] = 0
transactions_clean.loc[good_purchase & (transactions_clean["channel"] == "Online"), "online_count"] = 1
transactions_clean["failed_count"] = transactions_clean["is_failed"].astype(int)
transactions_clean["insufficient_count"] = transactions_clean["is_insufficient_balance"].astype(int)

print("Total spending: $", round(transactions_clean["purchase_value"].sum()))

Total spending: $ 246031455


In [18]:
# Monthly total for every card
card_month = transactions_clean.groupby(["card_id", "month_start"])[["purchase_count", "purchase_value", "refund_value",
                                                                    "online_count", "failed_count", "insufficient_count"]].sum()
card_month = card_month.reset_index()
card_month = card_month.rename(columns={"purchase_count": "transactions", "online_count": "online_transactions",
                                        "failed_count": "failed_transactions", "insufficient_count": "insufficient_balance"})
card_month.head()

,card_id,month_start,transactions,purchase_value,refund_value,online_transactions,failed_transactions,insufficient_balance
0,0,2010-01-01,25,1409.15,291.0,2,1,0
1,0,2010-02-01,13,1066.49,445.0,1,0,0
2,0,2010-03-01,23,2249.95,314.0,4,1,1
3,0,2010-04-01,26,1672.85,343.0,4,1,1
4,0,2010-05-01,29,3205.27,290.0,3,1,0


In [19]:
# A card keeps its balance even in months with no spending.
# So we make one row for every card and every month, from its first month until October 2019.
all_months = pd.date_range("2010-01-01", "2019-10-01", freq="MS")
first_month = card_month.groupby("card_id")["month_start"].min()

grid_cards = []
grid_months = []
for card_id in first_month.index:
    for month in all_months:
        if month >= first_month[card_id]:
            grid_cards.append(card_id)
            grid_months.append(month)

month_grid = pd.DataFrame({"card_id": grid_cards, "month_start": grid_months})

# Join the real monthly numbers. Months with no spending get 0
card_month = month_grid.merge(card_month, on=["card_id", "month_start"], how="left")
card_month = card_month.fillna(0)

print("Rows:", len(card_month))

Rows: 141900


In [20]:
# Add customer id, credit limit and year to every row
card_month = card_month.merge(cards_clean[["card_id", "customer_id", "credit_limit"]], on="card_id", how="left")
card_month["year"] = card_month["month_start"].dt.year

# Counts should be whole numbers
card_month["transactions"] = card_month["transactions"].astype(int)
card_month["online_transactions"] = card_month["online_transactions"].astype(int)
card_month["failed_transactions"] = card_month["failed_transactions"].astype(int)
card_month["insufficient_balance"] = card_month["insufficient_balance"].astype(int)

card_month.head()

,card_id,month_start,transactions,purchase_value,refund_value,online_transactions,failed_transactions,insufficient_balance,customer_id,credit_limit,year
0,0,2010-01-01,25,1409.15,291.0,2,1,0,1362,33900.0,2010
1,0,2010-02-01,13,1066.49,445.0,1,0,0,1362,33900.0,2010
2,0,2010-03-01,23,2249.95,314.0,4,1,1,1362,33900.0,2010
3,0,2010-04-01,26,1672.85,343.0,4,1,1,1362,33900.0,2010
4,0,2010-05-01,29,3205.27,290.0,3,1,0,1362,33900.0,2010


In [21]:
# PAYMENT BEHAVIOR (ESTIMATED)
# The data has no payments, so we estimate them from the real credit score and debt to income:
#   Full Payer:    credit score 740 or more and debt to income below 2. Pays 100% of the bill
#   Minimum Payer: credit score below 670 or debt to income 2.5 or more. Pays 10% of the bill
#   Partial Payer: everyone else. Pays 40% of the bill
customers_clean["payment_behavior"] = "Partial Payer"
customers_clean.loc[(customers_clean["credit_score"] >= 740) & (customers_clean["debt_to_income"] < 2), "payment_behavior"] = "Full Payer"
customers_clean.loc[(customers_clean["credit_score"] < 670) | (customers_clean["debt_to_income"] >= 2.5), "payment_behavior"] = "Minimum Payer"

customers_clean["payment_share"] = 0.4
customers_clean.loc[customers_clean["payment_behavior"] == "Full Payer", "payment_share"] = 1.0
customers_clean.loc[customers_clean["payment_behavior"] == "Minimum Payer", "payment_share"] = 0.1

card_month = card_month.merge(customers_clean[["customer_id", "payment_share"]], on="customer_id", how="left")

customers_clean["payment_behavior"].value_counts()

payment_behavior
Partial Payer    425
Full Payer       247
Minimum Payer    183
Name: count, dtype: int64

In [22]:
# Monthly balance for every card
#   bill (statement balance) = balance from last month + purchases - refunds
#   payment = bill x payment share
#   closing balance = bill - payment (this moves to next month)
# If the balance goes above the credit limit, the extra must be paid at once.
card_month = card_month.sort_values(["card_id", "month_start"]).reset_index(drop=True)

statement_list = []
payment_list = []
closing_list = []
balance = 0
last_card = None

for i in range(len(card_month)):
    card_id = card_month.loc[i, "card_id"]
    if card_id != last_card:
        balance = 0
        last_card = card_id

    bill = balance + card_month.loc[i, "purchase_value"] - card_month.loc[i, "refund_value"]
    if bill < 0:
        bill = 0

    payment = bill * card_month.loc[i, "payment_share"]
    limit = card_month.loc[i, "credit_limit"]
    if bill - payment > limit:
        payment = bill - limit

    balance = bill - payment

    statement_list.append(round(bill, 2))
    payment_list.append(round(payment, 2))
    closing_list.append(round(balance, 2))

card_month["statement_balance"] = statement_list
card_month["payment_amount"] = payment_list
card_month["closing_balance"] = closing_list

card_month.tail()

,card_id,month_start,transactions,purchase_value,refund_value,online_transactions,failed_transactions,insufficient_balance,customer_id,credit_limit,year,payment_share,statement_balance,payment_amount,closing_balance
141895,6131,2019-07-01,19,531.65,0.0,1,0,0,821,3900.0,2019,0.1,4233.74,423.37,3810.36
141896,6131,2019-08-01,15,371.27,0.0,3,0,0,821,3900.0,2019,0.1,4181.63,418.16,3763.47
141897,6131,2019-09-01,23,685.21,0.0,2,0,0,821,3900.0,2019,0.1,4448.68,548.68,3900.00
141898,6131,2019-10-01,21,1685.21,138.0,4,0,0,821,3900.0,2019,0.1,5447.21,1547.21,3900.00
141899,6144,2019-10-01,1,1127.60,0.0,1,0,0,737,6600.0,2019,1.0,1127.60,1127.60,0.00


In [23]:
# CUSTOMER TOTALS
customer_totals = card_month.groupby("customer_id")[["transactions", "purchase_value", "insufficient_balance"]].sum()
customer_totals = customer_totals.reset_index()
customer_totals = customer_totals.rename(columns={"transactions": "total_transactions", "purchase_value": "total_spend"})

# Number of cards and total credit limit
card_totals = cards_clean.groupby("customer_id")["credit_limit"].agg(["count", "sum"]).reset_index()
card_totals = card_totals.rename(columns={"count": "credit_cards", "sum": "total_credit_limit"})

# Last 12 months (November 2018 to October 2019): spending and utilization
last_12 = card_month[card_month["month_start"] >= "2018-11-01"]
last_12_totals = last_12.groupby("customer_id")[["statement_balance", "credit_limit", "purchase_value"]].sum().reset_index()
last_12_totals["utilization_pct"] = (last_12_totals["statement_balance"] / last_12_totals["credit_limit"] * 100).round(1)
last_12_totals["spend_last_12m"] = last_12_totals["purchase_value"].round(2)

# Outstanding balance at the end of October 2019
last_month = card_month[card_month["month_start"] == "2019-10-01"]
balance_totals = last_month.groupby("customer_id")["closing_balance"].sum().reset_index()
balance_totals = balance_totals.rename(columns={"closing_balance": "outstanding_balance"})

# Put everything into the customers table
customers_clean = customers_clean.merge(customer_totals, on="customer_id", how="left")
customers_clean = customers_clean.merge(card_totals, on="customer_id", how="left")
customers_clean = customers_clean.merge(last_12_totals[["customer_id", "utilization_pct", "spend_last_12m"]], on="customer_id", how="left")
customers_clean = customers_clean.merge(balance_totals, on="customer_id", how="left")

customers_clean["utilization_band"] = pd.cut(customers_clean["utilization_pct"], bins=[-1, 10, 30, 50, 80, 100000],
                                             labels=["Under 10%", "10% to 30%", "30% to 50%", "50% to 80%", "Above 80%"])

customers_clean[["customer_id", "total_spend", "total_credit_limit", "utilization_pct", "outstanding_balance"]].head()

,customer_id,total_spend,total_credit_limit,utilization_pct,outstanding_balance
0,825,535157.21,12400.0,82.6,5723.93
1,708,391576.25,98100.0,0.0,0.00
2,68,559033.25,25700.0,56.5,8659.70
3,1075,359267.09,11200.0,60.9,4808.38
4,1116,627904.23,24800.0,19.7,0.00


In [24]:
# RISK SCORE (more points = more risk)
customers_clean["risk_score"] = 0

# Credit score points
customers_clean.loc[customers_clean["credit_score_band"] == "Poor", "risk_score"] += 40
customers_clean.loc[customers_clean["credit_score_band"] == "Fair", "risk_score"] += 30
customers_clean.loc[customers_clean["credit_score_band"] == "Good", "risk_score"] += 15

# Debt to income points
customers_clean.loc[customers_clean["debt_to_income"] >= 2.5, "risk_score"] += 25
customers_clean.loc[(customers_clean["debt_to_income"] >= 1.5) & (customers_clean["debt_to_income"] < 2.5), "risk_score"] += 15

# Utilization points
customers_clean.loc[customers_clean["utilization_pct"] > 80, "risk_score"] += 25
customers_clean.loc[(customers_clean["utilization_pct"] > 50) & (customers_clean["utilization_pct"] <= 80), "risk_score"] += 15
customers_clean.loc[(customers_clean["utilization_pct"] > 30) & (customers_clean["utilization_pct"] <= 50), "risk_score"] += 5

# Declined for insufficient balance in more than 1% of transactions
decline_rate = customers_clean["insufficient_balance"] / customers_clean["total_transactions"] * 100
customers_clean.loc[decline_rate > 1, "risk_score"] += 10

# Risk level
customers_clean["risk_level"] = "Low Risk"
customers_clean.loc[customers_clean["risk_score"] >= 25, "risk_level"] = "Medium Risk"
customers_clean.loc[customers_clean["risk_score"] >= 50, "risk_level"] = "High Risk"

customers_clean["risk_level"].value_counts()

risk_level
Medium Risk    330
Low Risk       326
High Risk      199
Name: count, dtype: int64

In [25]:
# VALUE SEGMENT: top 10% spenders are High Value, the next 40% are Mid Value
high_value_limit = customers_clean["total_spend"].quantile(0.9)
mid_value_limit = customers_clean["total_spend"].quantile(0.5)

customers_clean["value_segment"] = "Low Value"
customers_clean.loc[customers_clean["total_spend"] >= mid_value_limit, "value_segment"] = "Mid Value"
customers_clean.loc[customers_clean["total_spend"] > high_value_limit, "value_segment"] = "High Value"

# ACTION GROUP: what should the bank do?
customers_clean["action_group"] = "Maintain"
customers_clean.loc[(customers_clean["risk_level"] == "Low Risk") & (customers_clean["utilization_pct"] < 30), "action_group"] = "Grow Usage"
customers_clean.loc[customers_clean["value_segment"] == "High Value", "action_group"] = "Retain"
customers_clean.loc[customers_clean["risk_level"] == "High Risk", "action_group"] = "Monitor Risk"

customers_clean["action_group"].value_counts()

action_group
Maintain        323
Grow Usage      269
Monitor Risk    199
Retain           64
Name: count, dtype: int64

In [26]:
# Yearly spending by merchant category and by location for every card
category_year = transactions_clean.groupby(["card_id", "year", "mcc"])[["purchase_count", "purchase_value"]].sum().reset_index()
category_year = category_year.rename(columns={"purchase_count": "transactions"})
category_year = category_year[category_year["transactions"] > 0]

location_year = transactions_clean.groupby(["card_id", "year", "merchant_state", "location_type"])[["purchase_count", "purchase_value"]].sum().reset_index()
location_year = location_year.rename(columns={"purchase_count": "transactions"})
location_year = location_year[location_year["transactions"] > 0]

# Merchant category names
category_names = pd.DataFrame(list(mcc_codes.items()), columns=["mcc", "category_name"])
category_names["mcc"] = category_names["mcc"].astype(int)

print("Category rows:", len(category_year))
print("Location rows:", len(location_year))

Category rows: 376173
Location rows: 82367


In [27]:
# STEP 4: ANSWER THE BUSINESS QUESTIONS
# Q1. What is the overall transaction volume and value?
total_transactions = card_month["transactions"].sum()
total_value = card_month["purchase_value"].sum()

print("Total transactions:", total_transactions)
print("Total value: $", round(total_value))
print("Average transaction: $", round(total_value / total_transactions, 2))
print("Customers:", len(customers_clean))
print()
print(transactions_clean.groupby("channel")["purchase_value"].sum().round())

Total transactions: 3829550
Total value: $ 246031455
Average transaction: $ 64.25
Customers: 855



channel
Chip       84586640.0
Online     34815838.0
Swipe     126628977.0
Name: purchase_value, dtype: float64


In [28]:
# Q2. Which customer segments spend the most?
print(customers_clean.groupby("age_group", observed=True)["total_spend"].sum().round())
print()
print(customers_clean.groupby("income_segment", observed=True)["total_spend"].sum().round())
print()
print(customers_clean.groupby("gender")["total_spend"].sum().round())

age_group
18 to 29     5530872.0
30 to 44    70520151.0
45 to 59    95102834.0
60 plus     74877597.0
Name: total_spend, dtype: float64

income_segment
Low Income              28426079.0
Middle Income          156599326.0
Upper Middle Income     50414294.0
High Income             10591757.0
Name: total_spend, dtype: float64

gender
Female    125182737.0
Male      120848718.0
Name: total_spend, dtype: float64


In [29]:
# Q3. Which merchant categories have the highest transaction value?
category_total = category_year.groupby("mcc")[["transactions", "purchase_value"]].sum().reset_index()
category_total = category_total.merge(category_names, on="mcc")

category_total.sort_values("purchase_value", ascending=False).head(10)

,mcc,transactions,purchase_value,category_name
43,4829,226434,21858368.25,Money Transfer
59,5541,329546,16603567.82,Service Stations
53,5300,201534,14748366.53,Wholesale Clubs
56,5411,442874,14685345.48,"Grocery Stores, Supermarkets"
74,5912,247882,13330472.70,Drug Stores and Pharmacies
45,4900,97644,12012756.83,"Utilities - Electric, Gas, Water, Sanitary"
57,5499,312357,11242108.79,Miscellaneous Food Stores
42,4814,87211,11010023.15,Telecommunication Services
55,5311,154076,10098957.42,Department Stores
90,7538,153461,8665233.06,Automotive Service Shops


In [30]:
# Q4. How does card usage change month over month?
monthly_trend = card_month.groupby("month_start")[["transactions", "purchase_value"]].sum()
monthly_trend["change_pct"] = (monthly_trend["purchase_value"].pct_change() * 100).round(1)
monthly_trend.tail(12)

,transactions,purchase_value,change_pct
month_start,,,
2018-11-01,32417,2094342.74,-2.7
2018-12-01,33345,2115855.53,1.0
2019-01-01,33844,2143797.14,1.3
2019-02-01,30305,1942872.75,-9.4
2019-03-01,33627,2179856.33,12.2
2019-04-01,32480,2076645.21,-4.7
2019-05-01,33809,2176717.59,4.8
2019-06-01,32855,2113550.00,-2.9
2019-07-01,33505,2137189.44,1.1


In [31]:
# Q4. Yearly trend (2019 has only 10 months)
yearly_trend = card_month.groupby("year")[["transactions", "purchase_value"]].sum()
yearly_trend["change_pct"] = (yearly_trend["purchase_value"].pct_change() * 100).round(1)
yearly_trend

,transactions,purchase_value,change_pct
year,,,
2010,366640,23380430.26,NaN
2011,381854,24482555.23,4.7
2012,385052,24706152.80,0.9
2013,390037,25201190.11,2.0
2014,392790,25265670.97,0.3
2015,395798,25406771.82,0.6
2016,392793,25460794.39,0.2
2017,398431,25585478.44,0.5
2018,396044,25453481.75,-0.5


In [32]:
# Q5. Which customers have the highest credit utilization?
print(customers_clean["utilization_band"].value_counts().sort_index())

customers_clean.sort_values("utilization_pct", ascending=False).head(10)[["customer_id", "credit_score", "total_credit_limit", "utilization_pct"]]

utilization_band
Under 10%     174
10% to 30%    278
30% to 50%    124
50% to 80%    104
Above 80%     175
Name: count, dtype: int64


,customer_id,credit_score,total_credit_limit,utilization_pct
25,643,697,500.0,631.0
619,1998,688,100.0,438.3
286,200,701,1400.0,312.7
126,1842,637,1300.0,281.3
491,1985,680,400.0,271.3
81,705,609,5200.0,252.2
720,1954,683,5900.0,239.7
265,1016,719,200.0,230.1
348,1555,717,2300.0,207.6
660,1291,699,1900.0,193.0


In [33]:
# Q6. How does credit limit relate to spending?
print("Correlation:", round(customers_clean["total_credit_limit"].corr(customers_clean["spend_last_12m"]), 2))

customers_clean.groupby(pd.cut(customers_clean["total_credit_limit"], bins=[0, 10000, 20000, 40000, 1000000]), observed=True)["spend_last_12m"].mean().round()

Correlation: 0.27


total_credit_limit
(0, 10000]          23321.0
(10000, 20000]      27124.0
(20000, 40000]      36567.0
(40000, 1000000]    46658.0
Name: spend_last_12m, dtype: float64

In [34]:
# Q7. Which customers have big outstanding balances?
print("Total outstanding balance: $", round(customers_clean["outstanding_balance"].sum()))

customers_clean.sort_values("outstanding_balance", ascending=False).head(10)[["customer_id", "total_credit_limit", "outstanding_balance", "payment_behavior"]]

Total outstanding balance: $ 3332244


,customer_id,total_credit_limit,outstanding_balance,payment_behavior
767,1648,36700.0,36700.00,Minimum Payer
48,1913,68400.0,35138.49,Minimum Payer
272,1307,96400.0,34336.61,Minimum Payer
36,985,29500.0,29329.14,Minimum Payer
243,1809,32500.0,28748.86,Minimum Payer
582,1500,28700.0,25771.78,Minimum Payer
216,1283,32300.0,24591.98,Minimum Payer
262,763,24300.0,24300.00,Minimum Payer
368,1163,49100.0,22742.72,Minimum Payer
317,1513,36100.0,22503.27,Minimum Payer


In [35]:
# Q8. What are the payment behavior patterns?
customers_clean.groupby("payment_behavior")[["credit_score", "utilization_pct", "outstanding_balance"]].mean().round(1)

,credit_score,utilization_pct,outstanding_balance
payment_behavior,,,
Full Payer,782.3,16.4,0.0
Minimum Payer,653.5,102.0,10783.1
Partial Payer,712.0,40.0,3197.5


In [36]:
# Q9. Which categories have the highest average transaction value? (only categories with 1,000+ transactions)
category_total["avg_value"] = (category_total["purchase_value"] / category_total["transactions"]).round(2)
big_categories = category_total[category_total["transactions"] >= 1000]

big_categories.sort_values("avg_value", ascending=False).head(10)[["category_name", "transactions", "avg_value"]]

,category_name,transactions,avg_value
8,"Tools, Parts, Supplies Manufacturing",1799,752.26
11,Leather Goods,1586,748.92
39,Airlines,1583,745.96
2,Steel Products Manufacturing,1824,742.76
1,Steelworks,1758,741.06
9,Miscellaneous Metals,1533,736.72
13,Upholstery and Drapery Stores,1619,732.36
15,Pottery and Ceramics,1604,728.46
14,"Brick, Stone, and Related Materials",1647,724.13
104,Hospitals,1923,715.87


In [37]:
# Q10. Which locations have the most card activity?
print(location_year.groupby("location_type")["purchase_value"].sum().round())

usa_only = location_year[location_year["location_type"] == "USA"]
usa_only.groupby("merchant_state")["purchase_value"].sum().sort_values(ascending=False).head(10).round()

location_type
International      2009014.0
Online            35099027.0
USA              208923414.0
Name: purchase_value, dtype: float64


merchant_state
CA    26730234.0
TX    16964107.0
NY    14622824.0
FL    12191317.0
PA     8455137.0
MI     7811928.0
OH     7767081.0
IL     7720939.0
NC     6986798.0
GA     6513583.0
Name: purchase_value, dtype: float64

In [38]:
# Q11. Which customer segments have higher credit risk?
customers_clean["is_high_risk"] = customers_clean["risk_level"] == "High Risk"

print((customers_clean.groupby("credit_score_band", observed=True)["is_high_risk"].mean() * 100).round(1))
print()
print((customers_clean.groupby("age_group", observed=True)["is_high_risk"].mean() * 100).round(1))
print()
print((customers_clean.groupby("payment_behavior")["is_high_risk"].mean() * 100).round(1))

credit_score_band
Poor         89.5
Fair         84.1
Good         16.3
Very Good     8.4
Excellent     4.9
Name: is_high_risk, dtype: float64

age_group
18 to 29    45.5
30 to 44    22.6
45 to 59    28.9
60 plus     15.8
Name: is_high_risk, dtype: float64

payment_behavior
Full Payer        0.4
Minimum Payer    85.8
Partial Payer     9.6
Name: is_high_risk, dtype: float64


In [39]:
# Q12. Does spending or payment behavior change over time?
yearly = card_month.groupby("year")[["purchase_value", "payment_amount"]].sum()
yearly["cards"] = card_month.groupby("year")["card_id"].nunique()
yearly["spend_per_card"] = (yearly["purchase_value"] / yearly["cards"]).round()
yearly["paid_pct"] = (yearly["payment_amount"] / yearly["purchase_value"] * 100).round(1)
yearly.round()

,purchase_value,payment_amount,cards,spend_per_card,paid_pct
year,,,,,
2010,23380430.0,18292530.0,1003,23310.0,78.0
2011,24482555.0,21706629.0,1102,22216.0,89.0
2012,24706153.0,22063479.0,1149,21502.0,89.0
2013,25201190.0,22468890.0,1193,21124.0,89.0
2014,25265671.0,22693786.0,1233,20491.0,90.0
2015,25406772.0,22800550.0,1272,19974.0,90.0
2016,25460794.0,22897756.0,1302,19555.0,90.0
2017,25585478.0,23032311.0,1322,19354.0,90.0
2018,25453482.0,22966492.0,1337,19038.0,90.0


In [40]:
# Q13. Which customers can be targeted to use their card more?
grow_usage = customers_clean[customers_clean["action_group"] == "Grow Usage"]

print("Grow Usage customers:", len(grow_usage))
print("Unused credit: $", round((grow_usage["total_credit_limit"] - grow_usage["outstanding_balance"]).sum()))
grow_usage.groupby("age_group", observed=True)["customer_id"].count()

Grow Usage customers: 269
Unused credit: $ 5347307


age_group
18 to 29      2
30 to 44     74
45 to 59     79
60 plus     114
Name: customer_id, dtype: int64

In [41]:
# Q14. What do high value customers look like?
customers_clean.groupby("value_segment")[["age", "yearly_income", "credit_score", "total_credit_limit", "total_spend", "utilization_pct"]].mean().round(1)

,age,yearly_income,credit_score,total_credit_limit,total_spend,utilization_pct
value_segment,,,,,,
High Value,52.0,64920.0,711.4,28307.0,762448.4,69.3
Low Value,54.5,37220.9,720.4,13748.0,147134.0,40.0
Mid Value,54.5,49008.5,721.2,20607.3,343961.0,48.8


In [42]:
# STEP 5: SAVE CLEAN TABLES FOR POWER BI (one Excel file, one sheet per table)
dim_customer = customers_clean[["customer_id", "gender", "age", "age_group", "yearly_income", "income_segment",
                                "credit_score", "credit_score_band", "total_debt", "debt_to_income", "debt_band",
                                "credit_cards", "total_credit_limit", "total_spend", "spend_last_12m", "utilization_pct",
                                "utilization_band", "outstanding_balance", "payment_behavior", "risk_score", "risk_level",
                                "value_segment", "action_group"]].round(2)

dim_card = cards_clean[["card_id", "customer_id", "card_brand", "has_chip", "num_cards_issued", "credit_limit", "limit_band", "open_year"]]

dim_year = pd.DataFrame({"year": range(2010, 2020)})

dim_date = pd.DataFrame({"month_start": all_months})
dim_date["year"] = dim_date["month_start"].dt.year
dim_date["quarter"] = "Q" + dim_date["month_start"].dt.quarter.astype(str)
dim_date["month_number"] = dim_date["month_start"].dt.month
dim_date["month_name"] = dim_date["month_start"].dt.strftime("%b")

fact_card_month = card_month[["card_id", "month_start", "transactions", "purchase_value", "refund_value", "online_transactions",
                              "failed_transactions", "insufficient_balance", "credit_limit", "statement_balance",
                              "payment_amount", "closing_balance"]].round(2)

fact_category = category_year.round(2)
fact_location = location_year.round(2)

C:\Users\sumay\AppData\Local\Temp\ipykernel_10832\3464048271.py:20: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  "payment_amount", "closing_balance"]].round(2)


In [43]:
# Write the Excel file (this takes a few minutes)
excel_writer = pd.ExcelWriter("CreditCard_Clean_Data.xlsx")

dim_customer.to_excel(excel_writer, sheet_name="dim_customer", index=False)
dim_card.to_excel(excel_writer, sheet_name="dim_card", index=False)
category_names.to_excel(excel_writer, sheet_name="dim_merchant_category", index=False)
dim_year.to_excel(excel_writer, sheet_name="dim_year", index=False)
dim_date.to_excel(excel_writer, sheet_name="dim_date", index=False)
fact_card_month.to_excel(excel_writer, sheet_name="fact_card_month", index=False)
fact_category.to_excel(excel_writer, sheet_name="fact_category", index=False)
fact_location.to_excel(excel_writer, sheet_name="fact_location", index=False)

excel_writer.close()
print("Saved CreditCard_Clean_Data.xlsx")

Saved CreditCard_Clean_Data.xlsx
